# ChakraXAttnUNet ? Kaggle dataset audit
Run the cells in order. This version audits mounted Kaggle data directly and does not clone a GitHub branch.


In [ ]:
from pathlib import Path
from collections import Counter
import json, cv2
ROOT = Path('/kaggle/input')
image_exts = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}
video_exts = {'.avi','.mp4','.mov','.mkv','.webm','.mpeg','.mpg'}
mask_words = {'mask','masks','groundtruth','ground_truth','segmentation','segmentations','label','labels','annotation','annotations'}
datasets = [p for p in ROOT.rglob('*') if p.is_dir() and len(p.relative_to(ROOT).parts) == 3 and any(x.is_file() for x in p.rglob('*'))]
reports = []
for dataset in sorted(datasets):
    files = [p for p in dataset.rglob('*') if p.is_file()]
    images = [p for p in files if p.suffix.lower() in image_exts]
    videos = [p for p in files if p.suffix.lower() in video_exts]
    masks = [p for p in images if any(w in p.as_posix().lower() for w in mask_words)]
    sizes = Counter()
    for p in images[:500]:
        im = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
        if im is not None: sizes[f'{im.shape[1]}x{im.shape[0]}'] += 1
    reports.append({'dataset': str(dataset), 'image_count': len(images), 'video_count': len(videos), 'mask_candidate_count': len(masks), 'top_resolutions': sizes.most_common(10), 'top_folders': Counter(str(p.parent.relative_to(dataset)) for p in files).most_common(10), 'sample_images': [str(p.relative_to(dataset)) for p in images[:10]], 'sample_masks': [str(p.relative_to(dataset)) for p in masks[:10]], 'sample_videos': [str(p.relative_to(dataset)) for p in videos[:10]]})
out = Path('/kaggle/working/dataset_audit'); out.mkdir(parents=True, exist_ok=True)
(out / 'summary.json').write_text(json.dumps(reports, indent=2))
print(json.dumps([{k: r[k] for k in ('dataset','image_count','video_count','mask_candidate_count')} for r in reports], indent=2))
print('Detailed report:', out / 'summary.json')


In [6]:
from pathlib import Path
import json

report = json.loads(Path('/kaggle/working/dataset_audit/summary.json').read_text())
for item in report:
    print(item['dataset'])
    print(' images:', item['image_count'], 'videos:', item['video_count'], 'mask candidates:', item['mask_candidate_count'])
    print(' folders:', item['top_folders'][:5])
    print(' image samples:', item['sample_images'][:3])
    print(' mask samples:', item['sample_masks'][:3])
    print()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/dataset_audit/summary.json'